# ETL — Orchestration du pipeline de préparation des données

Ce notebook **n'implémente plus la logique ETL directement** : il appelle le module
`etl_pipeline/` (situé dans ce même dossier `01_etl/`), qui contient tout le code de
nettoyage, de construction des dimensions et de la table de faits, organisé en étapes
séparées et testables indépendamment :

| Fichier du module | Rôle |
|---|---|
| `etl_pipeline/config.py` | Constantes partagées (chemins, date de référence, mappings métier) |
| `etl_pipeline/extract.py` | Lecture du fichier principal + des 8 fichiers `dim_*.xlsx` |
| `etl_pipeline/clean.py` | Nettoyage : doublons, valeurs manquantes, conversion des dates |
| `etl_pipeline/dimension_lookup.py` | Décodage des codes anonymisés via les dimensions réelles |
| `etl_pipeline/dimensions.py` | Construction des 6 tables de dimension du schéma en étoile |
| `etl_pipeline/fact.py` | Construction de `FACT_ACCOUNT_EVENT` |
| `etl_pipeline/load.py` | Chargement vers PostgreSQL (SQLAlchemy), échec non bloquant |
| `etl_pipeline/pipeline.py` | Orchestrateur : enchaîne toutes les étapes |

**Pourquoi un module plutôt qu'un notebook monolithique ?** Un notebook unique mélange
logique métier et exploration, ce qui rend le code difficile à réutiliser (par exemple
depuis `04_machine_learning`) et difficile à tester (impossible d'exécuter "juste le
nettoyage" sans rejouer toutes les cellules précédentes). Le module peut être importé
tel quel depuis n'importe quel autre notebook ou script du projet :

```python
from etl_pipeline.pipeline import run_pipeline
resultats = run_pipeline()
```

Ce notebook se contente d'appeler ce module, d'afficher ses résultats, et de produire
les KPIs métier (section 3).

## Schéma cible (validé en équipe)

| Table | Grain | Rôle |
|---|---|---|
| `DIM_CLIENT` | 1 ligne par `CUSTOMER_NO` | Attributs client (âge, KYC, statut marital, **secteur d'activité réel via dim_INDUSTRY**) |
| `DIM_DATE` | 1 ligne par jour calendaire | Dimension temps classique |
| `DIM_BRANCH` | 1 ligne par `BRANCH` | Attributs agence |
| `DIM_ACCOUNT` | 1 ligne par `ACCOUNT_NO` | Attributs compte (**catégorie et devise décodées via dim_CATEGORY_ACCOUNT / dim_CURRENCY**) |
| `DIM_PRODUCT` | 1 ligne par **contrat** (compte × produit) | Attributs contractuels — voir note de grain dans le README du module |
| `DIM_CLOSURE` | 1 ligne par motif de clôture | **Libellés réels et classification volontaire/involontaire via dim_Closure_reason** |
| `FACT_ACCOUNT_EVENT` | 1 ligne par événement compte/produit | Table de faits : mesures + `churn` |

### Nouveauté par rapport à la version précédente : intégration des 8 dimensions xlsx

Les fichiers `dim_*.xlsx` fournis par l'encadrant n'avaient jamais été réellement exploités
jusqu'ici. Ils sont maintenant intégrés partout où ils apportent une valeur réelle et
vérifiable :

- **`dim_INDUSTRY.xlsx`** confirme que le code `9000` (61,9% des lignes) signifie bien
  `"Other"` — hypothèse précédemment déduite par déduction statistique, maintenant
  vérifiée. Le code `9998` reste introuvable dans le référentiel (40 187 clients
  concernés) — gap réel à signaler, pas une erreur du pipeline.
- **`dim_Closure_reason.xlsx`** décode enfin les codes `BANK.REASON.N`, auparavant
  opaques. ⚠️ Ce fichier contient encore le nom réel de l'institution (sous la forme
  d'un acronyme) dans certains libellés texte — `etl_pipeline/dimension_lookup.py`
  le retire systématiquement avant toute utilisation, conformément à la règle
  d'anonymat de la documentation.
- **`dim_CATEGORY_ACCOUNT.xlsx`** et **`dim_CURRENCY.xlsx`** ajoutent des libellés
  lisibles aux codes de catégorie de compte et de devise dans `DIM_ACCOUNT`.

## 1. Exécution du pipeline

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Rend le module etl_pipeline importable depuis ce notebook, quel que soit le
# répertoire de travail courant de Jupyter (qui peut être 01_etl/ ou
# 01_etl/notebooks/ selon comment le notebook est lancé) : on remonte jusqu'à
# trouver le dossier etl_pipeline/ plutôt que de supposer un chemin relatif fixe.
_search_dir = Path.cwd()
for _ in range(4):
    if (_search_dir / "etl_pipeline").is_dir():
        sys.path.insert(0, str(_search_dir))
        break
    _search_dir = _search_dir.parent
else:
    raise ImportError(
        "Impossible de localiser le dossier etl_pipeline/ en remontant depuis "
        f"{Path.cwd()} — vérifiez que ce notebook est bien situé sous 01_etl/."
    )

from etl_pipeline.pipeline import run_pipeline

# load_to_db=True tente le chargement PostgreSQL (échoue proprement si aucun serveur
# n'est disponible, voir etl_pipeline/load.py) ; mettre False pour itérer plus vite
# sans dépendre d'une base.
resultats = run_pipeline(load_to_db=True)

16:06:19 | INFO     | etl_pipeline.pipeline | ======================================================================


16:06:19 | INFO     | etl_pipeline.pipeline | DÉBUT DU PIPELINE ETL


16:06:19 | INFO     | etl_pipeline.pipeline | ======================================================================


16:06:19 | INFO     | etl_pipeline.pipeline | --- ÉTAPE 1/5 : EXTRACT ---


16:06:22 | INFO     | etl_pipeline.extract | Extraction terminée : 528,883 lignes, 34 colonnes


...

16:06:34 | WARNING  | etl_pipeline.load | Chargement PostgreSQL de 'fact_account_event' impossible : OperationalError ((psycopg2.OperationalError) connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)). Le DataFrame reste disponible en mémoire ; seul le chargement en base a échoué.


16:06:34 | INFO     | etl_pipeline.load | Chargement PostgreSQL : 0/7 tables chargées avec succès.


16:06:34 | INFO     | etl_pipeline.pipeline | ======================================================================


16:06:34 | INFO     | etl_pipeline.pipeline | PIPELINE ETL TERMINÉ


16:06:34 | INFO     | etl_pipeline.pipeline | ======================================================================


## 2. Inspection des résultats

### 2.1 DataFrame nettoyé (grain événement, 1 ligne par ligne source après nettoyage)

In [2]:
df = resultats["df"]
print(f"Lignes : {len(df):,} | Colonnes : {df.shape[1]}")
df.head(3)

Lignes : 445,803 | Colonnes : 34


,CUSTOMER_NO,ACCOUNT_NO,NATIONALITY,RESIDENCE,MARITAL_STATUS,CUST_OPENING_DATE,DATE_OF_BIRTH,NATURE_CLIENT,BRANCH,SCORE_KYC,...,PRODUCT_LINE,PRODUCT,ACCOUNTNATURE,STARTDATE,MATURITYDATE,AMOUNT,FIXEDRATE,PRODUCT_STATUS,PARTYCLASS,LOB
0,C318650,A0365322,TN,TN,M,2004-09-30,1969-01-01,PPH,BR114,LR,...,LENDING,RT.RT.CRD.IMMOBILERS.527,Crédit acquisition logement TEGF6,2025-12-27,2029-06-27,10954600.0,4.5,CURRENT,Retail,4
1,C318648,A0373555,TN,TN,M,2004-09-30,1960-01-01,PPH,BR114,LR,...,DEPOSITS,BANK.CAT.NEG.SIM,DEPOTS A TERME,2026-01-02,NaT,0.0,0.0,UNAUTH,Retail,4
2,C318650,A0348290,TN,TN,M,2004-09-30,1969-01-01,PPH,BR114,LR,...,LENDING,RT.RT.CRD.IMMOBILERS.548,Crédit rénovation,2025-12-27,2038-05-27,113593077.0,4.5,CURRENT,Retail,4


### 2.2 Dimensions construites

In [3]:
for nom, table in resultats["dimensions"].items():
    print(f"{nom:15s} : {len(table):>8,} lignes, {table.shape[1]} colonnes")

dim_client      :  319,129 lignes, 16 colonnes
dim_branch      :      141 lignes, 6 colonnes
dim_account     :  410,587 lignes, 13 colonnes
dim_product     :  445,803 lignes, 11 colonnes
dim_closure     :       20 lignes, 6 colonnes
dim_date        :   43,879 lignes, 5 colonnes


In [4]:
dim_client = resultats["dimensions"]["dim_client"]
dim_client.head(3)

,client_key,CUSTOMER_NO,DATE_OF_BIRTH,age,MARITAL_STATUS,NATIONALITY,RESIDENCE,NATURE_CLIENT,PARTYCLASS,LOB,INDUSTRY,SCORE_KYC,COMPLETED_FILE,CUST_OPENING_DATE,LAST_REVIEW_DATE,INDUSTRY_LABEL
0,1,C295435,1983-01-01,43,C,TN,TN,PPH,Retail,4,9000,LR,NO,2004-09-30,2004-09-30,Other
1,2,C316033,1965-01-01,61,C,TN,TN,PPH,Retail,4,9000,H1,NO,2004-09-30,2004-09-30,Other
2,3,C286226,1992-01-01,34,C,TN,TN,PPH,Retail,4,9000,LR,NO,2004-09-30,2004-09-30,Other


In [5]:
dim_closure = resultats["dimensions"]["dim_closure"]
dim_closure

,closure_key,closure_reason,closure_label,is_voluntary,closure_category,churn_type
0,1,INCONNUE,"Clôturé, motif non documenté dans le système s...",None,INCONNU,INCONNU
1,2,BANK.REASON.13,Autre,None,INCONNU,INCONNU
2,3,BANK.REASON.9,Cessation d activite,True,INCONNU,INCONNU
3,4,NON_FERME,Compte actif (pas de clôture),False,INCONNU,NON_APPLICABLE
4,5,BANK.REASON.1,Changement d adresse de domicile,None,INCONNU,INCONNU
5,6,BANK.REASON.12,Client décdé,False,INCONNU,INCONNU
6,7,BANK.REASON.10,Ouverture d un nouveau compte aupres d une aut...,None,INCONNU,INCONNU
7,8,BANK.REASON.2,Tarification chere,True,INCONNU,INCONNU
8,9,BANK.REASON.6,Refus d octroi de credit,False,INCONNU,INCONNU
9,10,BANK.REASON.3,Changement d employeur,True,INCONNU,INCONNU


### 2.3 Table de faits

In [6]:
fact_account_event = resultats["fact_account_event"]
print(f"Lignes : {len(fact_account_event):,} | Colonnes : {fact_account_event.shape[1]}")
fact_account_event.head(3)

Lignes : 445,803 | Colonnes : 14


,client_key,account_key,product_key,branch_key,date_key,closure_key,acct_balance,salary,amount,fixedrate,acct_tenure_days,client_tenure_days,nb_accounts,churn
0,280670,1,1,1,34207,1,-10714.347,2725.739,10954600.0,4.5,2346.0,7937,4,1
1,280672,2,2,1,36530,1,0.000,3300.537,0.0,0.0,NaN,7937,6,1
2,280670,3,3,1,35592,1,-113033.101,2725.739,113593077.0,4.5,961.0,7937,4,1


### 2.4 Résultat du chargement PostgreSQL

`None` si `load_to_db=False`, sinon un dictionnaire `{nom_table: succès booléen}` —
voir `etl_pipeline/load.py` pour le détail du comportement en cas d'échec de connexion
(le pipeline continue sans planter, message clair affiché par table).

In [7]:
resultats["load_results"]

{'dim_client': False,
 'dim_branch': False,
 'dim_account': False,
 'dim_product': False,
 'dim_closure': False,
 'dim_date': False,
 'fact_account_event': False}

## 3. KPIs métier — recalculés sur les données réelles

Les chiffres ci-dessous sont calculés directement sur `df` (résultat de l'étape CLEAN),
pas recopiés d'une note antérieure.

In [8]:
print(f"KPI 01 - Taux de churn global (niveau evenement, apres nettoyage) :")
print(f"  {(df['ACCOUNT_STATUS']=='Closed').mean()*100:.1f}% (sur {len(df):,} lignes)")
print(f"\nRappel - taux de churn niveau compte (voir section 6.1 de etl_pipeline.fact) : "
      f"{fact_account_event.drop_duplicates(subset='account_key')['churn'].mean()*100:.1f}%")

KPI 01 - Taux de churn global (niveau evenement, apres nettoyage) :
  41.2% (sur 445,803 lignes)

Rappel - taux de churn niveau compte (voir section 6.1 de etl_pipeline.fact) : 36.1%


In [9]:
print("KPI 04 - Solde moyen par statut (ACCT_BALANCE) :")
print(df.groupby('ACCOUNT_STATUS')['ACCT_BALANCE'].agg(['mean', 'median']).round(2))

KPI 04 - Solde moyen par statut (ACCT_BALANCE) :


                    mean  median
ACCOUNT_STATUS                  
Active           -518.11   13.82
Closed          65321.79    0.00


In [10]:
print("KPI 05 - Salaire moyen par statut (SALARY) :")
print(df.groupby('ACCOUNT_STATUS')['SALARY'].mean().round(2))

KPI 05 - Salaire moyen par statut (SALARY) :


ACCOUNT_STATUS
Active    3663.49
Closed    8908.56
Name: SALARY, dtype: float64

In [11]:
print("KPI 06 - Taux de churn par segment client (PARTYCLASS) :")
print(
    df.groupby('PARTYCLASS')['ACCOUNT_STATUS']
      .apply(lambda x: (x == 'Closed').mean() * 100)
      .round(1)
      .sort_values(ascending=False)
)

KPI 06 - Taux de churn par segment client (PARTYCLASS) :


PARTYCLASS
Corporate          72.7
Elite              65.9
Retail             38.4
Corporate Small    33.3
Autre               0.0
Name: ACCOUNT_STATUS, dtype: float64


In [12]:
print("KPI 07 - Taux de churn par ligne de produit (PRODUCT_LINE) :")
print(
    df.groupby('PRODUCT_LINE')['ACCOUNT_STATUS']
      .apply(lambda x: (x == 'Closed').mean() * 100)
      .round(1)
      .sort_values(ascending=False)
)

KPI 07 - Taux de churn par ligne de produit (PRODUCT_LINE) :


PRODUCT_LINE
DEPOSITS            95.7
SANS_LIGNE          75.6
LENDING             40.3
SAFE.DEPOSIT.BOX    33.2
ACCOUNTS            12.2
Name: ACCOUNT_STATUS, dtype: float64


In [13]:
print("KPI 08 - Taux de churn par score KYC :")
kyc_order = ['LR', 'MR', 'H1', 'H2', 'H3']
print(
    df.groupby('SCORE_KYC')['ACCOUNT_STATUS']
      .apply(lambda x: (x == 'Closed').mean() * 100)
      .reindex(kyc_order)
      .round(1)
)

KPI 08 - Taux de churn par score KYC :


SCORE_KYC
LR    39.6
MR    35.2
H1    49.5
H2    57.3
H3    49.1
Name: ACCOUNT_STATUS, dtype: float64


In [14]:
print("KPI 09 (nouveau) - Taux de churn par secteur d'activite reel (via dim_INDUSTRY) :")
industry_labels = dim_client[['INDUSTRY', 'INDUSTRY_LABEL']].drop_duplicates()
df_with_label = df.merge(industry_labels, on='INDUSTRY', how='left')
top_industries = (
    df_with_label[df_with_label['INDUSTRY'] != '9000']  # exclut 'Other', dominant et peu informatif
      .groupby('INDUSTRY_LABEL')['ACCOUNT_STATUS']
      .apply(lambda x: (x == 'Closed').mean() * 100)
      .round(1)
)
counts = (
    df_with_label[df_with_label['INDUSTRY'] != '9000']
      .groupby('INDUSTRY_LABEL').size()
)
result = pd.DataFrame({'churn_rate_pct': top_industries, 'count': counts})
print(result[result['count'] >= 30].sort_values('churn_rate_pct', ascending=False).head(15))

KPI 09 (nouveau) - Taux de churn par secteur d'activite reel (via dim_INDUSTRY) :


                                                    churn_rate_pct  count
INDUSTRY_LABEL                                                           
construction aeronautique et spatiale                         99.2    356
fabrication tube,tuyaux,profiles creux et acces...            98.0    744
fonds de placement et entites financieres simil...            97.9    424
hebergement medicalise                                        96.5   1766
fabrication autres articles en papier ou en carton            96.4    641
fabrication aliments pour animaux de compagnie                96.0    174
fabrication autres articles a mailles                         96.0    100
edition autres logiciels                                      94.9     79
fabrication de papier ,carton ondules, emballag...            91.5    541
activites de banque centrale                                  90.6     53
fabrication articles en papier a usage sanitair...            90.3    247
commerce de gros de produits chimiques

In [15]:
print("KPI 10 (nouveau) - Taux de churn par motif de cloture reel (via dim_Closure_reason) :")
closure_with_label = dim_closure[dim_closure['closure_reason'].str.startswith('BANK.REASON')]
print(closure_with_label[['closure_reason', 'closure_label', 'is_voluntary']].to_string())

print("\nRepartition volontaire / involontaire parmi les comptes fermes avec motif connu :")
fact_with_closure = fact_account_event.merge(
    dim_closure[['closure_key', 'is_voluntary']], on='closure_key', how='left'
)
closed_only = fact_with_closure[fact_with_closure['churn'] == 1]
print(closed_only['is_voluntary'].value_counts(dropna=False))

KPI 10 (nouveau) - Taux de churn par motif de cloture reel (via dim_Closure_reason) :
     closure_reason                                                   closure_label is_voluntary
1    BANK.REASON.13                                                           Autre         None
2     BANK.REASON.9                                            Cessation d activite         True
4     BANK.REASON.1                                Changement d adresse de domicile         None
5    BANK.REASON.12                                                    Client décdé        False
6    BANK.REASON.10    Ouverture d un nouveau compte aupres d une autre agence BANK         None
7     BANK.REASON.2                                              Tarification chere         True
8     BANK.REASON.6                                        Refus d octroi de credit        False
9     BANK.REASON.3                                          Changement d employeur         True
10    BANK.REASON.5  Recherche service ou

**Colonnes à exclure des modèles ML** (data leakage ou absence de valeur prédictive) :
`ACCT_CLOSE_DATE` (n'existe qu'après le churn), `CLOSURE_REASON` (label post-churn),
`CUSTOMER_NO`/`ACCOUNT_NO` (identifiants), `NATIONALITY`/`RESIDENCE` (quasi-constantes à
plus de 93% TN — variance faible mais non nulle, à vérifier avant exclusion définitive).

## 4. Synthèse pour `04_machine_learning`

| Table | Grain | Clé de jointure |
|---|---|---|
| `fact_account_event` | 1 ligne par événement | `client_key`, `account_key`, `product_key`, `branch_key`, `closure_key`, `date_key` |
| `dim_client` | 1 ligne par client | `client_key` |
| `dim_account` | 1 ligne par compte | `account_key` |
| `dim_product` | 1 ligne par contrat | `product_key` |

Pour obtenir un jeu d'entraînement sans relancer tout ce notebook, importer directement
le module :

```python
import sys
sys.path.insert(0, "../01_etl")  # ou le chemin relatif approprié
from etl_pipeline.pipeline import run_pipeline

resultats = run_pipeline(load_to_db=False)  # False : pas besoin de PostgreSQL pour le ML
fact = resultats["fact_account_event"]
dim_client = resultats["dimensions"]["dim_client"]
```

Le label `churn` est déjà présent dans `fact_account_event` (propagé depuis le compte) —
utilisable directement comme variable cible.